# Оценка ТЗ: трудоёмкость и приоритет уточнений

Методология: `metodology.pdf`. Для каждого компонента ChatGPT оценивает распределение
`[Zero, Small, Medium, Large]`; при отсутствии содержательных данных Python берёт prior меню.
`d = k × [0,1,3,10]`, `E[D] = Σpᵢdᵢ`, `U = √Σpᵢ(dᵢ−E[D])²`,
`Raw Priority = U`, `Priority = U / √c`. Сортировка по Priority — по убыванию.

Вероятности являются экспертными оценками. Потенциально снимаемая неопределённость равна U
при допущении, что ответ полностью определит уровень компонента. Это не гарантированная
экономия труда; стандартные отклонения компонентов нельзя просто складывать.


In [ ]:
%pip install "openai>=1.40,<3" "python-dotenv>=1,<2" "pandas>=2.1,<3" "pydantic>=2.8,<3" "python-docx>=1.1,<2" "pypdf>=5,<7" "tqdm>=4.66,<5"

In [ ]:
from pathlib import Path
import sys

# Запуск из корня репозитория, evaluator или evaluator/evaluator_v2.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'evaluator' / 'evaluator_v2' / 'probability_evaluator.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Откройте ноутбук внутри репозитория case-finder.')
sys.path.insert(0, str(ROOT))
BASE = ROOT / 'evaluator' / 'evaluator_v2'
from evaluator.evaluator_v2.probability_evaluator import (
    load_menu, load_spec, evaluate, summarize, save_results, top_questions,
    EFFORT, UNCERTAINTY, QUESTION,
)
from IPython.display import display, Markdown
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI
import os

# Ближайший .env имеет приоритет; значения ключей не выводятся.
for env_path in (BASE / '.env', BASE.parent / '.env', ROOT / '.env'):
    load_dotenv(env_path, override=False)
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError('Добавьте OPENAI_API_KEY в .env в корне репозитория или evaluator_v2.')
MODEL = os.getenv('OPENAI_LLM_MODEL', 'gpt-4.1-mini')
client = OpenAI(timeout=120, max_retries=2)


In [ ]:
TZ_DIR = ROOT / 'evaluator' / 'TZ'
# Укажите файл ТЗ из evaluator/TZ; поддерживаются TXT, MD, DOCX и текстовый PDF.
TZ_PATH = TZ_DIR / 'example.docx'  # Замените на имя вашего ТЗ.
MENU_PATH = BASE / 'chatbots_menu.csv'
OUTPUT_DIR = BASE / 'results'
BATCH_SIZE = 6
MAX_SPEC_CHARS = 180_000  # Текст никогда не обрезается автоматически.
TOP_N = 15

print('Доступные ТЗ:')
for path in sorted(TZ_DIR.rglob('*')):
    if path.suffix.lower() in {'.txt', '.md', '.docx', '.pdf'}:
        print(path.relative_to(TZ_DIR))


In [ ]:
menu = load_menu(MENU_PATH)
text = load_spec(TZ_PATH, TZ_DIR)
print(f'Загружено {len(text):,} символов ТЗ; компонентов меню: {len(menu)}')
with tqdm(total=len(menu), desc='Анализ ТЗ', unit='комп.') as bar:
    def progress(done, total):
        bar.update(done - bar.n)
    results = evaluate(client, MODEL, menu, text, batch_size=BATCH_SIZE,
                       max_spec_chars=MAX_SPEC_CHARS, progress=progress)
csv_path = save_results(results, OUTPUT_DIR, TZ_PATH.stem)
print(f'CSV сохранён: {csv_path}')
display(results)


In [ ]:
print(f'Ожидаемые суммарные затраты: {results[EFFORT].sum():.2f} человекодней')
print(f'Компонентов, требующих уточнения: {results[QUESTION].str.strip().ne("").sum()}')
print('Потенциально снимаемая неопределённость указана для каждого вопроса отдельно.')
display(top_questions(results, TOP_N).style.format({
    'Priority': '{:.3f}',
    'Потенциально снимаемая неопределенность (человекодни)': '{:.2f}',
}))


In [ ]:
print('ChatGPT готовит общее заключение…')
conclusion = summarize(client, MODEL, text, results)
display(Markdown(conclusion))
report_path = OUTPUT_DIR / f'{TZ_PATH.stem}_заключение_v2.md'
report_path.write_text(conclusion, encoding='utf-8')
print(f'Заключение сохранено: {report_path}')
